# Real-Time Object Detection with YOLOv8

A computer vision project that detects and labels everyday objects in a video using a pre-trained **YOLOv8-nano** model, run on Google Colab's free T4 GPU.

**Pipeline:** upload a short video → load YOLOv8n → run inference frame-by-frame → download an annotated output video with bounding boxes, class labels, and confidence scores.

**Before you start:** Go to `Runtime > Change runtime type` and set Hardware accelerator to **T4 GPU**.

## 1. Install dependencies

`ultralytics` bundles the YOLOv8 architecture, pre-trained weights downloader, and inference utilities in one package. We install it fresh each Colab session since Colab environments don't persist.

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()  # confirms GPU is detected and package versions

## 2. Upload your test video

Run this cell and use the file picker to upload `input_video.mp4` (a 10-30 second clip works well). Alternatively, mount Google Drive if your video is stored there.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select your video file in the dialog
input_filename = list(uploaded.keys())[0]
print(f"Uploaded: {input_filename}")

**Alternative — mount Google Drive instead of uploading directly:**

```python
from google.colab import drive
drive.mount('/content/drive')
input_filename = '/content/drive/MyDrive/input_video.mp4'  # adjust path
```

## 3. Load the pre-trained YOLOv8-nano model

`yolov8n.pt` is the smallest/fastest pre-trained YOLOv8 checkpoint, trained on the COCO dataset (80 everyday object classes: person, car, dog, chair, etc.). The weights download automatically on first use.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print(model.names)  # the 80 COCO classes this model can detect

## 4. Run inference on the video

`model.predict()` runs the model frame-by-frame on the T4 GPU, drawing bounding boxes + class labels + confidence scores, and saves the annotated result.

- `save=True` writes the annotated video to `runs/detect/predict/`
- `conf=0.25` is the minimum confidence threshold to draw a detection (lower = more boxes, more false positives; raise it if the output looks noisy)
- Use `model.track(...)` instead of `model.predict(...)` if you want each object to keep a consistent ID across frames (useful for counting unique objects)

In [ ]:
results = model.predict(
    source=input_filename,
    save=True,
    conf=0.25,
    device=0  # use the GPU
)

print(f"Processed {len(results)} frames.")

## 5. Locate and download the annotated output

Colab saves results under an auto-incrementing folder (`predict`, `predict2`, ...). This cell finds the most recent one automatically.

In [ ]:
import glob, os

predict_dirs = sorted(glob.glob('runs/detect/predict*'), key=os.path.getmtime)
latest_dir = predict_dirs[-1]
output_files = glob.glob(f'{latest_dir}/*.mp4') + glob.glob(f'{latest_dir}/*.avi')
output_path = output_files[0]

print(f"Annotated video saved at: {output_path}")

files.download(output_path)  # triggers browser download

## 6. (Optional) Preview a few detections inline

Print per-frame detection summaries without downloading the full video first — useful for a quick sanity check.

In [ ]:
for i, r in enumerate(results[:5]):  # first 5 frames
    boxes = r.boxes
    detected = [model.names[int(c)] for c in boxes.cls] if boxes is not None else []
    print(f"Frame {i}: {detected}")

---
### Notes
- If you rerun cell 4, results accumulate in new `predict2/`, `predict3/`, ... folders — cell 5 always grabs the latest.
- To use your own trained weights later instead of the stock model, swap `YOLO('yolov8n.pt')` for `YOLO('path/to/best.pt')`.
- Resume framing: *"Built a real-time object detection pipeline using a pre-trained YOLOv8 model, running GPU-accelerated inference on Colab to detect and label objects in video, producing annotated output with bounding boxes and confidence scores."*